# Research Notebook — V6 Offline Engine
Replay historical Solana data → mine patterns → evaluate → propose live updates.

In [ ]:
import sys, os, json, asyncio, datetime as dt
ROOT = os.path.abspath('..')
if ROOT not in sys.path: sys.path.insert(0, ROOT)

from src.config import load_config
from src.research import storage, replay, evaluation, patterns, selector, integrate
from src.research.run import run_replay
from src.research.collectors import helius_collector, pool_collector

cfg = load_config()
print('helius_set=', bool(cfg.helius_api_key))

## 1. Collect 1 day of data
Edit dates and your `data/smart_wallets.json` first.

In [ ]:
from pathlib import Path
wallets = json.loads(Path('../data/smart_wallets.json').read_text()) if Path('../data/smart_wallets.json').exists() else []
print('wallets:', len(wallets))

start = dt.datetime(2026, 4, 30, 0, 0, tzinfo=dt.timezone.utc).timestamp()
end   = dt.datetime(2026, 5, 1, 0, 0, tzinfo=dt.timezone.utc).timestamp()

if wallets and cfg.helius_api_key:
    out = asyncio.get_event_loop().run_until_complete(
        helius_collector.collect_addresses(cfg.helius_api_key, wallets[:5], start, end, set(wallets))
    )
    print(out['total'])
else:
    print('skipping live collection — no wallets or no API key')

## 2. Run the replay

In [ ]:
summary = run_replay(
    start_ts=start, end_ts=end,
    capital_sol=0.667,
    smart_wallets=wallets,
    seed=42,
)
print(json.dumps(summary, indent=2))
run_db = Path(summary['run_dir']) / 'run.db'

## 3. Evaluate

In [ ]:
rep = evaluation.report(str(run_db))
print(json.dumps(rep, indent=2, default=str))

## 4. Mine patterns

In [ ]:
wc = patterns.mine_wallet_candidates(str(run_db), start, end)
pc = patterns.mine_precursors(start, end)
fp = patterns.cluster_fingerprints(str(run_db))
print('wallet_candidates:', len(wc))
print('precursors      :', len(pc))
print('fingerprint buckets:', len(fp))
out = patterns.write_outputs(Path('../data/research'), wc, pc, fp)
print(out)

## 5. Selection gate

In [ ]:
sel = selector.evaluate_run(str(run_db), capital_sol=0.667)
print('promoted:', sel.promoted, 'rank:', round(sel.rank_score, 3))
print('failures:', sel.failures)
selector.write_selection(Path('../data/research'), summary['run_id'], sel)

## 6. Optional: re-fit channel weights from this run

In [ ]:
weights = integrate.fit_lr_weights(str(run_db), lambda_ridge=0.5)
if weights:
    integrate.write_proposal(Path('../data/research/proposed_lr_weights.json'), weights)
    integrate.append_evolution_log({'source': f'research_run_{summary["run_id"]}', 'kind': 'weight_proposal', 'weights': weights})
print(weights or 'no channels logged in this run yet')